In [1]:
from pathlib import Path

from src.minbpe import RegexTokenizer
from src.gpt import GPTLanguageModel

import torch

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer_dir = Path("data") / "tokenizer"
checkpoint_dir = Path("data") / "ch05_checkpoints"

In [3]:
tokenizer = RegexTokenizer()
tokenizer.load(model_file=str(tokenizer_dir / "tokenizer.model"))

#ckpt_files = sorted(
#    checkpoint_dir.glob("checkpoint_*.pt"),
#    key=lambda x: x.stat().st_ctime,
#    #key=lambda x: int(x.name.split("-")[1]),
#    reverse=True,
#)
#checkpoint_path = ckpt_files[0]

checkpoint_path = checkpoint_dir / "checkpoint_000065.pt"

print(f"load checkpoint: {checkpoint_path}")
checkpoint = torch.load(checkpoint_path, weights_only=True, map_location=device)
parameters = checkpoint['meta']['parameters']

load checkpoint: data/ch05_checkpoints/checkpoint_000065.pt


In [4]:
model = GPTLanguageModel(
    vocab_size=parameters['vocab_size'],
    block_size=parameters['block_size'],
    n_embd=parameters['n_embd'],
    n_head=parameters['n_head'],
    n_layer=parameters['n_layer'],
    dropout=parameters['dropout'],
    ignore_index=tokenizer.special_tokens["<|padding|>"],
    device=device,
)

model = torch.compile(model)
model.load_state_dict(checkpoint["model_state_dict"])

num_parameters = sum(p.numel() for p in model.parameters()) / 1e6
print(f'--> {num_parameters:_.3}M parameters')
# print_model_structure(model)

_ = model.eval()

--> 13.8M parameters


In [6]:
def into_tokens(role: str, content: str) -> torch.Tensor:
    d = tokenizer.special_tokens
    #print("~~~", d)

    input_msg = f"<|startoftext|>{role}<|separator|>{content}<|endoftext|>"
    print(f"--> {role} message: {input_msg}")

    input_tokens = tokenizer.encode(input_msg, allowed_special="all")

    return torch.tensor(input_tokens, dtype=torch.long).unsqueeze(0).to(device)


def ask_llm(input_tokens):
    model_answer = ""

    while True:
        output_tokens = model.generate(input_tokens=input_tokens, max_new_tokens=1)
        last_generated_token = output_tokens[0, -1].item()

        #print("~~~", input_tokens[0])
        model_answer += tokenizer.decode([last_generated_token])

        if last_generated_token == tokenizer.special_tokens["<|endoftext|>"]:
            break

        input_tokens = torch.cat((input_tokens, output_tokens[:, -1:]), dim=1)

        #if len(output_tokens[0]) > parameters['block_size']:
        #    break
        if len(input_tokens[0]) > parameters['block_size']:
            input_tokens = input_tokens[:, -parameters['block_size']:]

    return model_answer

def ask_llm_advanced(input_tokens):
    model_answer = ""

    while True:
        output_tokens = model.advanced_generation(
            input_tokens=input_tokens, max_new_tokens=1,
            temperature=.9, top_k=50, top_p=None,
        )

        last_generated_token = output_tokens[0, -1].item()

        #print("~~~", input_tokens[0])
        model_answer += tokenizer.decode([last_generated_token])

        if last_generated_token == tokenizer.special_tokens["<|endoftext|>"]:
            break

        input_tokens = torch.cat((input_tokens, output_tokens[:, -1:]), dim=1)

        #if len(output_tokens[0]) > parameters['block_size']:
        #    break
        if len(input_tokens[0]) > parameters['block_size']:
            input_tokens = input_tokens[:, -parameters['block_size']:]

    return model_answer

In [6]:
user_content = "What's ChatGPT?"
input_tokens = into_tokens("user", user_content)

with torch.no_grad():
    output = model.generate(input_tokens=input_tokens, max_new_tokens=256)
    print("<-- assistant message:", tokenizer.decode(output[0].tolist()))

--> user message: <|startoftext|>user<|separator|>What's ChatGPT?<|endoftext|>
<-- assistant message: <|startoftext|>user<|separator|>What's ChatGPT?<|endoftext|><|startoftext|>assistant<|separator|>ChatGPT is a powerful and highly interpersonal assistant that can understand and respond to natural language inputs and provide information to the best of my abilities.<|endoftext|><|startoftext|>user<|separator|>But it also doesn't have a very powerful and thought-provoking interpretation.  Also, I can help you give it a try.<|endoftext|><|startoftext|>assistant<|separator|>Unlike ChatGPT I can implement this type of self-proactive and with common sources. However, it should be noted that some opinions restrict the top 8 students of ChatGPT may associate people with OpenAI's ChatGPT, while others are more well-equipped.<|endoftext|><|startoftext|>assistant<|separator|>How can I assist you?<|endoftext|><|startoftext|>assistant<|separator|>Sin, I am glad you ask.<|endoftext|><|startoftext|>u

In [7]:
user_content = "What's ChatGPT?"
input_tokens = into_tokens("user", user_content)

answer = ask_llm(input_tokens)
print(f"<-- assistant message: {answer}")

--> user message: <|startoftext|>user<|separator|>What's ChatGPT?<|endoftext|>
<-- assistant message: <|startoftext|>assistant<|separator|>what a meme? ChatGPT is a capitalized combination of advancements in music engineering, particularly in the field of artificial intelligence. It has a rich history, dating back to the past, and according to the public, has made it an excellent citing research and debate about society.<|endoftext|>


In [8]:
user_content = "What's ChatGPT?"
input_tokens = into_tokens("user", user_content)

answer = ask_llm(input_tokens)
print(f"<-- assistant message: {answer}")

--> user message: <|startoftext|>user<|separator|>What's ChatGPT?<|endoftext|>
<-- assistant message: <|startoftext|>assistant<|separator|>ChatGPT is a large language model (LLM) developed by OpenAI, which stands for "https://arxiv.org/presidents/LLMs.html), while OpenAI emplies a large language model like GPT-3.5. It is built on the InstructGPT paper from OpenAI, and is a company with a Discord server.

The goal of OpenAI is to provide an open model that can be run on an individuals based on their own prompts. By using these models, the algorithm can improve access to more vast amounts of text data in a wide range of industries.<|endoftext|>


In [7]:
user_content = "What's ChatGPT?"
input_tokens = into_tokens("user", user_content)

answer = ask_llm_advanced(input_tokens)
print(f"<-- assistant message: {answer}")

--> user message: <|startoftext|>user<|separator|>What's ChatGPT?<|endoftext|>
<-- assistant message: <|startoftext|>assistant<|separator|>ChatGPT is a large language model (LLM) developed by OpenAI, based on the GPT-3 architecture. It is designed to understand and answer text from a vast amount of data and can be used to generate human-like responses to text-based inputs. However, I cannot exhibit that such models are ordered in any ways similar to OpenAI, as well as the exact method of accomplishing ChatGPT with the quality and quality of data.<|endoftext|>


In [10]:
user_content = "什么是 ChatGPT?"
input_tokens = into_tokens("user", user_content)

answer = ask_llm(input_tokens)
print(f"<-- assistant message: {answer}")

--> user message: <|startoftext|>user<|separator|>什么是 ChatGPT?<|endoftext|>
<-- assistant message: <|startoftext|>assistant<|separator|>I'm sorry, but I don't have information on the topic of the essay. However, I can provide you with more information.

- I can provide you with information on the topic of the IP you are trying to confirm that I have found in. My interest is that I may look elsewhere with my output. Please let me know if you have any other questions.<|endoftext|>
